# Dense Dataset Reproduction: YOLO-Master-v0.1-N and YOLO-Master-EsMoE-N

This notebook trains the two nano YOLO-Master variants on dense detection datasets using built-in dataset configs such as `ultralytics/cfg/datasets/VisDrone.yaml`, `ultralytics/cfg/datasets/SKU-110K.yaml`, and `ultralytics/cfg/datasets/GlobalWheat2020.yaml`.

- `YOLO-Master-v0.1-N` -> `ultralytics/cfg/models/master/v0_1/det/yolo-master-n.yaml`
- `YOLO-Master-EsMoE-N` -> `ultralytics/cfg/models/master/v0/det/yolo-master-n.yaml`

Change `config.selected_dataset` and `config.selected_model` to switch datasets and models directly.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import sys
from dataclasses import dataclass, asdict
from pathlib import Path

import pandas as pd

def is_repo_root(path: Path) -> bool:
    """Return whether ``path`` looks like the YOLO-Master repository root.

    Args:
        path: Candidate directory to inspect.

    Returns:
        True when the expected repository markers are present.
    """
    return (path / "ultralytics").exists() and ((path / "examples").exists() or (path / "scripts").exists())


def find_repo_root(start: Path) -> Path:
    """Walk upward from ``start`` until the repository root is found.

    Args:
        start: Starting path for the upward search.

    Returns:
        Resolved repository root path.

    Raises:
        FileNotFoundError: If no repository root can be found.
    """
    probe = start.resolve()
    while True:
        if is_repo_root(probe):
            return probe
        if probe == probe.parent:
            break
        probe = probe.parent
    raise FileNotFoundError("Could not locate YOLO-Master repo root from current working directory.")


def default_device() -> str:
    """Pick the default Ultralytics device string for this notebook.

    Returns:
        ``"0"`` when CUDA is available, otherwise ``"cpu"``.
    """
    try:
        import torch
        return "0" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("YOLO_CONFIG_DIR", str(ROOT / ".tmp" / "ultralytics"))
Path(os.environ["YOLO_CONFIG_DIR"]).mkdir(parents=True, exist_ok=True)

from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

settings.update({"wandb": False})

ISSUE_DIR = ROOT / "scripts" / "issue49"
RUNS_DIR = Path(os.environ.get("YOLO_MASTER_RUNS_DIR", str(ISSUE_DIR / "runs"))).expanduser().resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root: {ROOT}")
print(f"python: {sys.executable}")
print(f"issue dir: {ISSUE_DIR}")
print(f"runs dir: {RUNS_DIR}")

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    """Static metadata describing a selectable model variant.

    Attributes:
        name: Human-readable model name shown in summaries.
        cfg: YAML config path passed to ``YOLO``.
        uses_esmoe: Whether the model contains ES-MoE modules.
    """
    name: str
    cfg: Path
    uses_esmoe: bool


@dataclass(frozen=True)
class DatasetSpec:
    """Static metadata describing a selectable dataset.

    Attributes:
        name: Human-readable dataset name.
        slug: Filesystem-safe dataset identifier.
        yaml: Ultralytics dataset YAML path.
    """
    name: str
    slug: str
    yaml: Path


@dataclass
class TrainConfig:
    """Notebook-level training configuration.

    Attributes:
        epochs: Total training epochs.
        imgsz: Input image size.
        batch: Batch size.
        device: Ultralytics device string.
        workers: Dataloader worker count.
        seed: Reproducibility seed.
        patience: Early stopping patience.
        amp: Whether to enable mixed precision.
        dense_eval_for_esmoe: Force dense inference during eval for ES-MoE runs.
        lr0: Optional override for the initial learning rate.
        use_wandb: Whether to enable W&B integration.
        wandb_project: W&B project name.
        wandb_entity: Optional W&B entity.
        wandb_mode: W&B mode, such as ``online``, ``offline``, or ``disabled``.
        wandb_group: W&B group label.
        selected_dataset: Dataset key from ``DATASET_SPECS``.
        selected_model: Model key from ``MODEL_SPECS``.
        run_tag: Optional explicit run tag.
        enable_lora: Opt into repo-wide LoRA defaults when True.
    """
    epochs: int = 100
    imgsz: int = 640
    batch: int = 16
    device: str = default_device()
    workers: int = 4
    seed: int = 42
    patience: int = 100
    amp: bool = True
    dense_eval_for_esmoe: bool = False
    lr0: float | None = None
    use_wandb: bool = True
    wandb_project: str = "yolo_master_issue49"
    wandb_entity: str | None = None
    wandb_mode: str = "online"  # online | offline | disabled
    wandb_group: str = "visdrone"
    selected_dataset: str = "VisDrone"
    selected_model: str = "YOLO-Master-v0.1-N"
    run_tag: str = ""  # set explicitly, or leave empty to auto-assign run001/run002/...
    enable_lora: bool = False

    def wandb_enabled(self) -> bool:
        """Return whether this config should attach W&B callbacks.

        Returns:
            True when W&B logging is enabled for the run.
        """
        return self.use_wandb and self.wandb_mode != "disabled"


DATASET_SPECS = {
    "VisDrone": DatasetSpec(
        name="VisDrone",
        slug="visdrone",
        yaml=ROOT / "ultralytics" / "cfg" / "datasets" / "VisDrone.yaml",
    ),
    "SKU-110K": DatasetSpec(
        name="SKU-110K",
        slug="sku_110k",
        yaml=ROOT / "ultralytics" / "cfg" / "datasets" / "SKU-110K.yaml",
    ),
    "GlobalWheat2020": DatasetSpec(
        name="GlobalWheat2020",
        slug="globalwheat2020",
        yaml=ROOT / "ultralytics" / "cfg" / "datasets" / "GlobalWheat2020.yaml",
    ),
}
MODEL_SPECS = {
    "YOLO-Master-v0.1-N": ModelSpec(
        name="YOLO-Master-v0.1-N",
        cfg=ROOT / "ultralytics" / "cfg" / "models" / "master" / "v0_1" / "det" / "yolo-master-n.yaml",
        uses_esmoe=False,
    ),
    "YOLO-Master-EsMoE-N": ModelSpec(
        name="YOLO-Master-EsMoE-N",
        cfg=ROOT / "ultralytics" / "cfg" / "models" / "master" / "v0" / "det" / "yolo-master-n.yaml",
        uses_esmoe=True,
    ),
}

config = TrainConfig()
config

## Optional: login to Weights & Biases

For public live logs, run `wandb login` in your shell before training. If you do not want remote logging, set:

- `config.use_wandb = False`, or
- `config.wandb_mode = "offline"`, or
- `config.wandb_mode = "disabled"`

In [ ]:
# Optional: execute this only when you want online W&B logging.
import wandb
wandb.login(relogin=True)

In [ ]:
METRIC_COLUMNS = [
    "epoch",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "train/box_loss",
    "train/cls_loss",
    "train/moe_loss",
    "val/box_loss",
    "val/cls_loss",
    "val/moe_loss",
]

SUMMARY_COLUMNS = [
    "dataset",
    "model",
    "run_name",
    "dense_eval",
    *METRIC_COLUMNS,
    "run_dir",
    "results_csv",
    "best_pt",
    "last_pt",
]


def get_selected_dataset(cfg: TrainConfig) -> DatasetSpec:
    """Return the dataset selected by ``cfg``.

    Args:
        cfg: Active training configuration.

    Returns:
        Dataset metadata for the selected dataset key.
    """
    return DATASET_SPECS[cfg.selected_dataset]


def get_selected_model(cfg: TrainConfig) -> ModelSpec:
    """Return the model selected by ``cfg``.

    Args:
        cfg: Active training configuration.

    Returns:
        Model metadata for the selected model key.
    """
    return MODEL_SPECS[cfg.selected_model]


def ensure_dataset(dataset_spec: DatasetSpec) -> dict:
    """Verify that a detection dataset is available locally.

    Args:
        dataset_spec: Dataset metadata to validate.

    Returns:
        Parsed dataset metadata from Ultralytics.
    """
    return check_det_dataset(str(dataset_spec.yaml), autodownload=True)


def build_train_overrides(cfg: TrainConfig) -> dict:
    """Return runtime overrides that keep baseline training stable by default.

    Args:
        cfg: Active training configuration.

    Returns:
        Mapping of keyword overrides passed to ``model.train``.
    """
    overrides = {}

    if cfg.enable_lora:
        return {"lr0": cfg.lr0} if cfg.lr0 is not None else {}

    overrides = {
        "lora_r": 0,
        "lora_save_adapters": False,
        "molora_num_experts": 0,
    }
    if cfg.lr0 is not None:
        overrides["lr0"] = cfg.lr0
    return overrides


def slugify(text: str) -> str:
    """Convert free-form text into a filesystem-safe slug.

    Args:
        text: Source text.

    Returns:
        Lowercase slug containing letters, digits, and underscores.
    """
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def next_run_tag(project_dir: Path) -> str:
    """Return the next sequential ``runNNN`` tag for ``project_dir``.

    Args:
        project_dir: Directory that stores per-run subdirectories.

    Returns:
        Next available run tag.
    """
    existing = []
    pattern = re.compile(r"^run(\d{3})$")
    if project_dir.exists():
        for path in project_dir.iterdir():
            if not path.is_dir():
                continue
            suffix = path.name.rsplit("_", 1)[-1]
            match = pattern.match(suffix)
            if match:
                existing.append(int(match.group(1)))
    next_idx = max(existing, default=0) + 1
    return f"run{next_idx:03d}"


def read_last_row(results_csv: Path) -> dict[str, str]:
    """Read the last metrics row from an Ultralytics ``results.csv`` file.

    Args:
        results_csv: Path to the training results CSV.

    Returns:
        Last CSV row as a dictionary, or an empty dict when unavailable.
    """
    if not results_csv.exists():
        return {}
    with results_csv.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return rows[-1] if rows else {}


def to_float(value):
    """Best-effort conversion to ``float``.

    Args:
        value: Input value from CSV parsing.

    Returns:
        Parsed float or ``None`` when conversion fails.
    """
    if value in (None, ""):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def should_use_wandb(cfg: TrainConfig) -> bool:
    """Return whether custom W&B callbacks should be registered.

    Args:
        cfg: Active training configuration.

    Returns:
        True when W&B logging is enabled.
    """
    return cfg.wandb_enabled()


def build_run_identity(model_slug: str, dataset_spec: DatasetSpec, run_tag: str, project_dir: Path) -> tuple[str, str]:
    """Build the stable run naming components used by this notebook.

    Args:
        model_slug: Filesystem-safe model identifier.
        dataset_spec: Selected dataset metadata.
        run_tag: Optional user-provided run tag.
        project_dir: Per-model run directory.

    Returns:
        Tuple of ``(resolved_run_tag, run_name)``.
    """
    resolved_run_tag = slugify(run_tag) if run_tag.strip() else next_run_tag(project_dir)
    run_name = f"{dataset_spec.slug}_{model_slug}_{resolved_run_tag}"
    return resolved_run_tag, run_name


def _custom_table(wandb, x, y, classes, title="Precision Recall Curve", x_title="Recall", y_title="Precision"):
    """Create a W&B area-under-curve table from interpolated series.

    Args:
        wandb: Imported ``wandb`` module.
        x: X-axis values.
        y: Y-axis values.
        classes: Class labels matching ``x``/``y`` values.
        title: Chart title.
        x_title: X-axis label.
        y_title: Y-axis label.

    Returns:
        W&B plot table object.
    """
    import polars as pl
    import polars.selectors as cs

    df = pl.DataFrame({"class": classes, "y": y, "x": x}).with_columns(cs.numeric().round(3))
    data = df.select(["class", "y", "x"]).rows()
    fields = {"x": "x", "y": "y", "class": "class"}
    string_fields = {"title": title, "x-axis-title": x_title, "y-axis-title": y_title}
    return wandb.plot_table(
        "wandb/area-under-curve/v0",
        wandb.Table(data=data, columns=["class", "y", "x"]),
        fields=fields,
        string_fields=string_fields,
    )


def _plot_curve(run, wandb, x, y, names=None, curve_id="precision-recall", title="Precision Recall Curve", x_title="Recall", y_title="Precision", num_x=100, only_mean=False):
    """Log a PR-style curve to W&B using interpolated points.

    Args:
        run: Active W&B run.
        wandb: Imported ``wandb`` module.
        x: Original X-axis values.
        y: Per-class Y-axis arrays.
        names: Optional per-class names.
        curve_id: W&B metric key.
        title: Chart title.
        x_title: X-axis label.
        y_title: Y-axis label.
        num_x: Interpolated sample count.
        only_mean: Whether to log only the mean curve.
    """
    import numpy as np

    if names is None:
        names = []
    x_new = np.linspace(x[0], x[-1], num_x).round(5)
    x_log = x_new.tolist()
    y_log = np.interp(x_new, x, np.mean(y, axis=0)).round(3).tolist()

    if only_mean:
        table = wandb.Table(data=list(zip(x_log, y_log)), columns=[x_title, y_title])
        run.log({title: wandb.plot.line(table, x_title, y_title, title=title)})
    else:
        classes = ["mean"] * len(x_log)
        for i, yi in enumerate(y):
            x_log.extend(x_new)
            y_log.extend(np.interp(x_new, x, yi))
            classes.extend([names[i]] * len(x_new))
        run.log({curve_id: _custom_table(wandb, x_log, y_log, classes, title, x_title, y_title)}, commit=False)


def _log_plots(run, wandb, plots, step, processed_plots):
    """Log newly generated plot images once per timestamp.

    Args:
        run: Active W&B run.
        wandb: Imported ``wandb`` module.
        plots: Plot registry from the trainer.
        step: Current training step.
        processed_plots: Cache of already logged plot timestamps.
    """
    for name, params in plots.copy().items():
        timestamp = params["timestamp"]
        if processed_plots.get(name) != timestamp:
            run.log({name.stem: wandb.Image(str(name))}, step=step)
            processed_plots[name] = timestamp


def make_dense_eval_callback():
    """Build a callback that forces dense inference for ES-MoE evaluation.

    Returns:
        Trainer callback that disables sparse inference on ES-MoE modules.
    """
    from ultralytics.nn.modules.moe.modules import ES_MOE
    from ultralytics.utils import LOGGER

    state = {"logged": False}

    def _callback(trainer):
        targets = []
        if getattr(trainer, "model", None) is not None:
            targets.append(trainer.model)
        ema = getattr(trainer, "ema", None)
        if ema is not None and getattr(ema, "ema", None) is not None:
            targets.append(ema.ema)

        changed = 0
        for target in targets:
            for module in target.modules():
                if isinstance(module, ES_MOE):
                    module.use_sparse_inference = False
                    changed += 1

        if changed and not state["logged"]:
            LOGGER.info(f"[issue49] dense eval enabled for {changed} ES_MOE modules")
            state["logged"] = True

    return _callback


def wandb_init_custom(state: dict, run_name: str, model_spec: ModelSpec, cfg: TrainConfig, dense_eval: bool, resolved_run_tag: str):
    """Initialize a W&B run for this notebook.

    Args:
        state: Mutable callback state.
        run_name: Final run name.
        model_spec: Selected model metadata.
        cfg: Active training configuration.
        dense_eval: Whether dense ES-MoE evaluation is enabled.
        resolved_run_tag: Final resolved run tag.
    """
    from ultralytics.utils import LOGGER

    if not should_use_wandb(cfg):
        return
    dataset_spec = get_selected_dataset(cfg)
    try:
        import wandb
    except Exception as exc:
        LOGGER.warning(f"[issue49] wandb unavailable: {exc}")
        return

    try:
        tags = [
            "issue49",
            dataset_spec.slug,
            slugify(cfg.wandb_group),
            slugify(model_spec.name),
            slugify(resolved_run_tag),
            "dense-eval" if dense_eval else "sparse-eval",
        ]
        state["wandb"] = wandb
        state["run"] = wandb.init(
            project=cfg.wandb_project,
            entity=cfg.wandb_entity,
            group=cfg.wandb_group,
            name=run_name,
            tags=tags,
            mode=cfg.wandb_mode,
            reinit=True,
            config={
                "dataset": dataset_spec.name,
                "dataset_slug": dataset_spec.slug,
                "wandb_group": cfg.wandb_group,
                "data": str(dataset_spec.yaml.relative_to(ROOT)),
                "model_name": model_spec.name,
                "model_cfg": str(model_spec.cfg.relative_to(ROOT)),
                "run_name": run_name,
                "run_tag": resolved_run_tag,
                "dense_eval_for_esmoe": dense_eval,
                **asdict(cfg),
            },
        )
        LOGGER.info(f"[issue49] wandb url: {getattr(state['run'], 'url', None)}")
    except Exception as exc:
        LOGGER.warning(f"[issue49] wandb init failed: {exc}")
        state["run"] = None


def wandb_log_ultralytics_defaults(state: dict, trainer, stage: str):
    """Mirror Ultralytics' default W&B logging behavior.

    Args:
        state: Mutable callback state.
        trainer: Active Ultralytics trainer.
        stage: Callback stage identifier.
    """
    from ultralytics.utils import LOGGER
    from ultralytics.utils.torch_utils import model_info_for_loggers

    run = state["run"]
    wandb = state["wandb"]
    if run is None or wandb is None:
        return

    step = int(getattr(trainer, "epoch", 0)) + 1
    if stage == "train_epoch_end":
        try:
            run.log(trainer.label_loss_items(trainer.tloss, prefix="train"), step=step)
            run.log(trainer.lr, step=step)
        except Exception as exc:
            LOGGER.warning(f"[issue49] wandb default train log failed at epoch {step}: {exc}")
        if trainer.epoch == 1:
            try:
                _log_plots(run, wandb, trainer.plots, step=step, processed_plots=state["processed_plots"])
            except Exception as exc:
                LOGGER.warning(f"[issue49] wandb default train plot log failed at epoch {step}: {exc}")
    elif stage == "fit_epoch_end":
        try:
            _log_plots(run, wandb, trainer.plots, step=step, processed_plots=state["processed_plots"])
            _log_plots(run, wandb, trainer.validator.plots, step=step, processed_plots=state["processed_plots"])
            if trainer.epoch == 0:
                run.log(model_info_for_loggers(trainer), step=step)
            run.log(trainer.metrics, step=step, commit=True)
        except Exception as exc:
            LOGGER.warning(f"[issue49] wandb default fit log failed at epoch {step}: {exc}")
    elif stage == "train_end":
        try:
            _log_plots(run, wandb, trainer.validator.plots, step=step, processed_plots=state["processed_plots"])
            _log_plots(run, wandb, trainer.plots, step=step, processed_plots=state["processed_plots"])
        except Exception as exc:
            LOGGER.warning(f"[issue49] wandb default final plot log failed: {exc}")
        try:
            art = wandb.Artifact(type="model", name=f"run_{run.id}_model")
            if trainer.best.exists():
                art.add_file(trainer.best)
                run.log_artifact(art, aliases=["best"])
        except Exception as exc:
            LOGGER.warning(f"[issue49] wandb default artifact log failed: {exc}")
        try:
            if trainer.args.plots and hasattr(trainer.validator.metrics, "curves_results"):
                for curve_name, curve_values in zip(trainer.validator.metrics.curves, trainer.validator.metrics.curves_results):
                    x, y, x_title, y_title = curve_values
                    _plot_curve(
                        run,
                        wandb,
                        x,
                        y,
                        names=list(trainer.validator.metrics.names.values()),
                        curve_id=f"curves/{curve_name}",
                        title=curve_name,
                        x_title=x_title,
                        y_title=y_title,
                    )
        except Exception as exc:
            LOGGER.warning(f"[issue49] wandb default curve log failed: {exc}")


def wandb_finish_custom(state: dict):
    """Finish the active W&B run and clear callback state.

    Args:
        state: Mutable callback state.
    """
    run = state["run"]
    if run is not None:
        try:
            run.finish()
        finally:
            state["run"] = None
            state["wandb"] = None


def make_wandb_callbacks(run_name: str, model_spec: ModelSpec, cfg: TrainConfig, dense_eval: bool, resolved_run_tag: str):
    """Build the W&B callback mapping expected by ``YOLO.add_callback``.

    Args:
        run_name: Final run name.
        model_spec: Selected model metadata.
        cfg: Active training configuration.
        dense_eval: Whether dense ES-MoE evaluation is enabled.
        resolved_run_tag: Final resolved run tag.

    Returns:
        Event-to-callback mapping.
    """
    state = {"run": None, "wandb": None, "processed_plots": {}}

    def on_train_start(trainer):
        wandb_init_custom(state, run_name, model_spec, cfg, dense_eval, resolved_run_tag)

    def on_train_epoch_end(trainer):
        wandb_log_ultralytics_defaults(state, trainer, "train_epoch_end")

    def on_fit_epoch_end(trainer):
        wandb_log_ultralytics_defaults(state, trainer, "fit_epoch_end")

    def on_train_end(trainer):
        wandb_log_ultralytics_defaults(state, trainer, "train_end")
        wandb_finish_custom(state)

    return {
        "on_train_start": on_train_start,
        "on_train_epoch_end": on_train_epoch_end,
        "on_fit_epoch_end": on_fit_epoch_end,
        "on_train_end": on_train_end,
    }


def train_one(model_spec: ModelSpec, cfg: TrainConfig) -> dict:
    """Train one selected model on one selected dataset.

    Args:
        model_spec: Selected model metadata.
        cfg: Active training configuration.

    Returns:
        Run summary dictionary containing paths and final metrics.
    """
    dataset_spec = get_selected_dataset(cfg)
    train_overrides = build_train_overrides(cfg)
    model_slug = slugify(model_spec.name.replace('.', ''))
    project_dir = RUNS_DIR / dataset_spec.slug / model_slug
    project_dir.mkdir(parents=True, exist_ok=True)
    tag_slug, run_name = build_run_identity(model_slug, dataset_spec, cfg.run_tag, project_dir)

    dense_eval = bool(model_spec.uses_esmoe and cfg.dense_eval_for_esmoe)
    model = YOLO(str(model_spec.cfg))

    if dense_eval:
        dense_cb = make_dense_eval_callback()
        model.add_callback("on_pretrain_routine_end", dense_cb)
        model.add_callback("on_train_start", dense_cb)

    if should_use_wandb(cfg):
        for event, callback in make_wandb_callbacks(run_name, model_spec, cfg, dense_eval, tag_slug).items():
            model.add_callback(event, callback)

    print(json.dumps({
        "dataset": dataset_spec.name,
        "run_name": run_name,
        "model": model_spec.name,
        "cfg": str(model_spec.cfg.relative_to(ROOT)),
        "data": str(dataset_spec.yaml.relative_to(ROOT)),
        "dense_eval": dense_eval,
        "epochs": cfg.epochs,
        "imgsz": cfg.imgsz,
        "batch": cfg.batch,
        "device": cfg.device,
        "enable_lora": cfg.enable_lora,
        "train_overrides": train_overrides,
    }, indent=2))

    model.train(
        data=str(dataset_spec.yaml),
        epochs=cfg.epochs,
        imgsz=cfg.imgsz,
        batch=cfg.batch,
        device=cfg.device,
        workers=cfg.workers,
        seed=cfg.seed,
        deterministic=False,
        project=str(project_dir),
        name=run_name,
        exist_ok=True,
        pretrained=False,
        val=True,
        plots=True,
        patience=cfg.patience,
        amp=cfg.amp,
        verbose=True,
        **train_overrides,
    )

    run_dir = project_dir / run_name
    results_csv = run_dir / "results.csv"
    last_row = read_last_row(results_csv)
    summary = {
        "dataset": dataset_spec.name,
        "dataset_slug": dataset_spec.slug,
        "model": model_spec.name,
        "run_name": run_name,
        "run_tag": tag_slug,
        "run_dir": str(run_dir),
        "results_csv": str(results_csv),
        "best_pt": str(run_dir / "weights" / "best.pt"),
        "last_pt": str(run_dir / "weights" / "last.pt"),
        "dense_eval": dense_eval,
    }
    for key in METRIC_COLUMNS:
        summary[key] = to_float(last_row.get(key))

    with (run_dir / "issue49_summary.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, ensure_ascii=False)

    summary_path = ISSUE_DIR / f"summary_{run_name}.csv"
    with summary_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=[c for c in SUMMARY_COLUMNS if c in summary])
        writer.writeheader()
        writer.writerow({k: summary.get(k) for k in writer.fieldnames})
    summary["summary_csv"] = str(summary_path)
    return summary


def summarize_runs(items: list[dict]) -> pd.DataFrame:
    """Convert one or more run summaries into a compact dataframe.

    Args:
        items: Run summary dictionaries returned by ``train_one``.

    Returns:
        Dataframe limited to the most relevant columns.
    """
    if not items:
        return pd.DataFrame()
    frame = pd.DataFrame(items)
    return frame[[c for c in SUMMARY_COLUMNS if c in frame.columns]]


In [ ]:
# Download / verify the selected dataset using the built-in dataset config.
dataset_spec = get_selected_dataset(config)
dataset_info = ensure_dataset(dataset_spec)
print(f"{dataset_spec.name} dataset is ready.")

In [ ]:
# Edit config here before launching training.
config.selected_dataset = "VisDrone"  # e.g. "VisDrone", "SKU-110K", or "GlobalWheat2020"
config.epochs = 100
config.imgsz = 640
config.batch = 16
config.device = default_device()
config.workers = 4
config.lr0 = None  # e.g. 0.001 to override the default base learning rate
config.amp = True
config.selected_model = "YOLO-Master-v0.1-N"  # or "YOLO-Master-EsMoE-N"
config.run_tag = ""  # e.g. "exp_bs16_e100"; leave empty to auto-assign run001/run002/...

# Keep False for as-shipped reproduction. Set True if you explicitly want dense eval for EsMoE.
config.dense_eval_for_esmoe = False

# LoRA settings.
config.enable_lora = False  # set True only when you explicitly want a LoRA experiment

# W&B settings.
config.use_wandb = True
config.wandb_project = "yolo_master_issue49"
config.wandb_entity = None
config.wandb_mode = "online"
config.wandb_group = "visdrone"   # e.g. "visdrone" or "sku110k"

assert config.selected_dataset in DATASET_SPECS, f"Unknown dataset: {config.selected_dataset}"
assert config.selected_model in MODEL_SPECS, f"Unknown model: {config.selected_model}"
config

In [ ]:
# Default: run exactly one model selected in config.selected_model.
summary = train_one(get_selected_model(config), config)
summary_df = summarize_runs([summary])
summary_df

In [ ]:
summary_path = ISSUE_DIR / f"summary_{summary['run_name']}.csv"
summary_df.to_csv(summary_path, index=False)
print(summary_path)
summary_df